In [1]:
#   ------------------------------------
#   Libraries
#   ------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import random
import math

#   For land/ocean coordinates identification
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import shapely.geometry as sgeom

#   For visualizations
import folium
import geopy.distance

#   For Cellular Automata
import mesa
from mesa import Model, Agent 
from mesa.space import MultiGrid
from mesa.experimental.cell_space import OrthogonalMooreGrid
from mesa.visualization import SolaraViz, make_space_component

from matplotlib.colors import Normalize, BoundaryNorm, LinearSegmentedColormap, to_hex
from matplotlib.cm import ScalarMappable
from enum import Enum

In [2]:
#   ------------------------------------
#   Weather Data
#   ------------------------------------
weather_df = pd.read_csv('hurricane_milton.csv')
weather_df['valid_time'] = pd.to_datetime(weather_df['valid_time'])

In [3]:
print(weather_df.shape)

(45936, 21)


In [4]:
#   ------------------------------------
#   Enumerators
#   ------------------------------------
class TemperatureFactor(Enum):
    """Using constants for now, needs to be calculated somehow."""
    OCEAN_HEATING = 0.1     # Heating factor for cells over the ocean
    OCEAN_COOLING = 0.1
    LAND_HEATING = 0.5
    LAND_COOLING = 0.5      # Cooling factor for cells over land

    @classmethod
    def apply_wind_factor(cls, total_wind_speed, k=0.05):
        """
        Dictionary of temperature change factors adjusted based on given total wind speed.
        Using linear scaling for simplicity.
        :param total_wind_speed: The total wind speed affecting temperature change.
        :param k: A scaling constant that determines how much wind increases cooling.
        :return: A dictionary with adjusted factors.
        """
        if total_wind_speed is None or math.isnan(total_wind_speed):
            #  Return the original enumerator if no valid wind speed is provided
            return {
                cls.OCEAN_HEATING: cls.OCEAN_HEATING.value,
                cls.OCEAN_COOLING: cls.OCEAN_COOLING.value,
                cls.LAND_HEATING: cls.LAND_HEATING.value,
                cls.LAND_COOLING: cls.LAND_COOLING.value,
            }
        
        wind_factor = 1 + k * total_wind_speed  # Linear scaling

        return {
            cls.OCEAN_HEATING: cls.OCEAN_HEATING.value * wind_factor,
            cls.OCEAN_COOLING: cls.OCEAN_COOLING.value * wind_factor,
            cls.LAND_HEATING: cls.LAND_HEATING.value * wind_factor,
            cls.LAND_COOLING: cls.LAND_COOLING.value * wind_factor,
        }
    
class WeatherParameter(Enum):
    """   Use numbers to identify each weather parameter."""
    TEMPERATURE = 0
    PRESSURE = 1
    WIND = 2
    PRECIPITATION = 3
    HUMIDITY = 4
    TERRAIN = 5

class WindSpeedFactor(Enum):
    OCEAN = 1.2  # Wind speeds up 20% over water
    LAND = 0.8   # Wind slows down 20% over land
    BASE_WIND_VARIATION = 0.5  # mph
    @classmethod
    def apply_wind_adjustment(cls, is_ocean, windspeed):
        return windspeed * (cls.OCEAN.value if is_ocean else cls.LAND.value)

In [ ]:
#   ------------------------------------
#   Functions
#   ------------------------------------

# Function to check if a point is ocean or land
def is_ocean_cell(lat, lon):
    """Check if given lat/long coordinates correspond to a cell over land or ocean."""
    # Create a Shapely Point object for the given coordinates
    point = sgeom.Point(lon, lat)
    
    # Use Cartopy to add land and ocean features
    land = cfeature.LAND
    ocean = cfeature.OCEAN

    # Loop through features and check if the point is inside the geometry
    for feature in [land, ocean]:
        # Cartopy features are available through cartopy.feature
        for geom in feature.geometries():
            if geom.contains(point):
                if feature == land:
                    return False  # The point is on land
                elif feature == ocean:
                    return True  # The point is in the ocean

    return False  # Default return if no match

def determine_grid_size(df, target_density=1.0):
    """
    Determines an appropriate grid size based on the number of data points.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing 'latitude' and 'longitude' columns.
        target_density (float): Approximate number of data points per grid cell (default 1.0).
    
    Returns:
        int: Optimal grid size (number of cells along one axis).
    """
    num_points = len(df)
    grid_size = int(np.sqrt(num_points / target_density))  # Approximate square grid
    
    return max(grid_size, 10)  # Ensure a minimum grid size

<h1>Cell Agent</h1>

Represents a small (31km by 31km) area within the region, which has states described by weather parameters: temperature, wind speed, wind direction, humidity, dewpoint, pressure, precipitation, precipitation type, cloud cover, terrain, terrain type. 

In [6]:
class MapGridCell(Agent):
    """An agent with fixed initial wealth."""

    @property
    def x(self):
        return self.cell.coordinate[0]

    @property
    def y(self):
        return self.cell.coordinate[1]
    
    @property
    def neighbors(self):
        #   Not including the center cell
        return self.model.grid.get_neighbors(
            self.pos, moore=True, include_center=False
        )
    
    def __init__(self, model, is_ocean, latitude, longitude):
        # Pass the parameters to the parent class.
        super().__init__(model)
        self.state = np.nan
        self.next_state = np.nan

        #   Using a dictionary to keep all the weather parameters on each
        #   step
        self.weather_data = {
            "temperature": np.nan,          # in degrees Celsius
            "totalwindspeed": np.nan,       # in m/h
            "humidity": np.nan,             # in percentage
            "pressure": np.nan,
            "precipitation": np.nan,
            "dewpoint": np.nan,
            "windspeednorth": np.nan,
            "windspeedeast": np.nan
        }

        self.next_weather_data = {
            "temperature": np.nan,          # in degrees Celsius
            "totalwindspeed": np.nan,       # in m/h
            "humidity": np.nan,             # in percentage
            "pressure": np.nan,
            "precipitation": np.nan,
            "dewpoint": np.nan,
            "windspeednorth": np.nan,       # in m/h
            "windspeedeast": np.nan         # in m/h
        }

        #   These attributes are used as additional information now
        self.simulation_parameter = model.simulation_parameter      #   Indicates which parameter is being simulated
        self.is_ocean = is_ocean
        self.latitude = latitude
        self.longitude = longitude

    def determine_state(self):
        """Compute the new value of the specified weather parameter inside 
        the cell at the next tick.  This is based on the values in the 
        neighbors.  The state is not changed here, but is just computed 
        and stored in self._nextState, because our current state may still 
        be necessary for our neighbors to calculate their next state.
        """
        self.update_next_parameters()
        if self.simulation_parameter == WeatherParameter.TEMPERATURE:
            self.next_state = self.next_weather_data["temperature"]
        if self.simulation_parameter == WeatherParameter.PRESSURE:
            self.next_state = self.next_weather_data["pressure"]
        if self.simulation_parameter == WeatherParameter.HUMIDITY:
            self.next_state = self.next_weather_data["humidity"]
        if self.simulation_parameter == WeatherParameter.PRECIPITATION:
            self.next_state = self.next_weather_data["precipitation"]
        if self.simulation_parameter == WeatherParameter.WIND:
            self.next_state = self.next_weather_data["totalwindspeed"]

    def assume_state(self):
        """Set the state to the new computed state -- computed in step()."""
        self.state = self.next_state
        self.weather_data["temperature"] = self.next_weather_data["temperature"]
        self.weather_data["windspeednorth"] = self.next_weather_data["windspeednorth"]
        self.weather_data["windspeedeast"] = self.next_weather_data["windspeedeast"]
        self.weather_data["totalwindspeed"] = self.next_weather_data["totalwindspeed"]
        self.weather_data["humidity"] = self.next_weather_data["humidity"]
        self.weather_data["pressure"] = self.next_weather_data["pressure"]
        self.weather_data["precipitation"] = self.next_weather_data["precipitation"]
        #   Call to reset all next values
        self.reset_next_state()

    #   -----------------------------
    #   Helper Methods for Cell Class
    #   -----------------------------
    def initialize_state(self):
        """Set the current state to value from data."""
        if self.simulation_parameter == WeatherParameter.TEMPERATURE:
            self.state = self.weather_data["temperature"]
        if self.simulation_parameter == WeatherParameter.PRESSURE:
            self.state = self.weather_data["pressure"]
        if self.simulation_parameter == WeatherParameter.HUMIDITY:
            self.state = self.weather_data["humidity"]
        if self.simulation_parameter == WeatherParameter.PRECIPITATION:
            self.state = self.weather_data["precipitation"]
        if self.simulation_parameter == WeatherParameter.WIND:
            self.state = self.weather_data["totalwindspeed"]
    
    def update_next_parameters(self):
        """Set the parameters to the new computed value."""
        self.next_weather_data["temperature"] = self.update_temperature()
        next_wind_data = self.update_wind_speed()
        if isinstance(next_wind_data, list) and len(next_wind_data) > 2:
            self.next_weather_data["windspeednorth"] = next_wind_data[0]
            self.next_weather_data["windspeedeast"] = next_wind_data[1]
            self.next_weather_data["totalwindspeed"] = next_wind_data[2]
        self.next_weather_data["humidity"] = self.update_humidity()
        self.next_weather_data["pressure"] = self.update_pressure()
        self.next_weather_data["precipitation"] = self.update_precipitation()

    def reset_next_state(self):
        """Reset the next state."""
        self.next_state = np.nan
        self.next_weather_data["temperature"] = np.nan
        self.next_weather_data["windspeednorth"] = np.nan
        self.next_weather_data["windspeedeast"] = np.nan
        self.next_weather_data["totalwindspeed"] = np.nan
        self.next_weather_data["humidity"] = np.nan
        self.next_weather_data["pressure"] = np.nan
        self.next_weather_data["precipitation"] = np.nan

    def update_temperature(self):
        neighbors = self.neighbors
        #   Get temperature factors adjusted based on wind speed
        #   Basic Rule 1:  Wind speed and direction affect how temperature is transferred
        #   from one cell to another.  For simplicity, we will just consider increasing the
        #   heating/cooling factor with the total wind speed.  Meaning that the stronger
        #   the winds, the greater the influence of neighboring cells in the current cell's
        #   temperature changes.
        temp_factors = TemperatureFactor.apply_wind_factor(self.weather_data["totalwindspeed"])
        OCEAN_HEATING = temp_factors[TemperatureFactor.OCEAN_HEATING]
        LAND_HEATING = temp_factors[TemperatureFactor.LAND_HEATING]
        OCEAN_COOLING = temp_factors[TemperatureFactor.OCEAN_COOLING]
        LAND_COOLING = temp_factors[TemperatureFactor.LAND_COOLING]

        #   Calculate average temperature of neighbors
        avg_temp = np.nanmean([neighbor.weather_data["temperature"] for neighbor in neighbors])
        #   Calculate average temperature of neighbors
        avg_pressure = np.nanmean([neighbor.weather_data["pressure"] for neighbor in neighbors])
        #   Calculate average temperature of neighbors
        avg_humidity = np.nanmean([neighbor.weather_data["humidity"] for neighbor in neighbors])

        temp_result = self.weather_data["temperature"]      #   No Change

        # CHANGES BASED ON OTHER TEMPERATURES AROUND
        # ------------------------------------------  
        # Update the temperature by comparing current cell's value
        # with the average of neighbors
        if self.weather_data["temperature"] < avg_temp:
            #   Apply heating factor based on cell being over the ocean or land
            #   Basic Rule 2:  If average temperature of neighbor cells is greater, the current cell
            #   can absorb heat and warm up.
            temp_result = self.weather_data["temperature"] + OCEAN_HEATING if self.is_ocean else self.weather_data["temperature"] + LAND_HEATING
        elif self.weather_data["temperature"] > avg_temp:
            #   Apply cooling factor based on cell being over the ocean or land
            #   Basic Rule 3:  If average temperature of neighbor cells is cooler, temperature
            #   should decrease.
            temp_result = self.weather_data["temperature"] - OCEAN_COOLING if self.is_ocean else self.weather_data["temperature"] - LAND_COOLING
        #   For now, do not change temperature if average is the same            

        # CHANGES BASED ON PRESSURE
        # -------------------------
        if self.weather_data["pressure"] < avg_pressure:
            #   Apply same heating factor for now
            #   Basic Rule 4:  If average pressure of neighbor cells is greater, temperature
            #   should increase.  This is because the lower pressure might indicate rising warm
            #   air.
            temp_result = temp_result + OCEAN_HEATING if self.is_ocean else temp_result + LAND_HEATING
        elif self.weather_data["pressure"] > avg_pressure:
            #   Apply cooling factor for now
            #   Basic Rule 5:  If average pressure of neighbor cells is lower, temperature
            #   should decrease since higher pressure in current cell might indicate descending cool air.
            temp_result = temp_result - OCEAN_COOLING if self.is_ocean else temp_result - LAND_COOLING
        #   For now, do not change temperature if average is the same    

        # CHANGES BASED ON HUMIDITY
        # -------------------------
        if self.weather_data["humidity"] > avg_humidity:
            #   Apply same heating factor for now
            #   Basic Rule 6:  If the humidity of the current cell is higher than the average neighbor humidity, temperature
            #   may increase because of the moisture's heat-retaining properties.
            temp_result = temp_result + OCEAN_HEATING if self.is_ocean else temp_result + LAND_HEATING
        elif self.weather_data["humidity"] < avg_humidity:
            #   Apply cooling factor for now
            #   Basic Rule 7:  If the humidity of the current cell is lower than the average neighbor humidity, temperature
            #   may decrease simulating the cooler, drier air conditions.
            temp_result = temp_result - OCEAN_COOLING if self.is_ocean else temp_result - LAND_COOLING
        #   For now, do not change temperature if average is the same

        return temp_result

    def update_pressure(self):
        """
        Update the pressure of the current cell based on temperature changes and wind convergence/divergence.
        """   
        #if air temperature increases, pressure should decrease. If temp decreases, pressure increases. But by how much ?
        #Winds that converge (meet) force air to rise, decreasing surface pressure. Winds that diverge (spread apart) cause air to sink, increasing surface pressure

        neighbors = self.neighbors

        #Retrieves the current temperature and pressure of the cell before any updates.
        current_temp = self.weather_data["temperature"]
        new_pressure = self.weather_data["pressure"]

        #Collects the temperature values from all neighboring cells.
        #Computes the average temperature of these neighbors.
        neighbor_temps = [cell.weather_data["temperature"] for cell in neighbors]
        avg_neighbor_temp = np.nansum(neighbor_temps) / len(neighbor_temps)
    
        #Finds the difference between the average neighbor temperature and the cell’s current temperature.
        temp_difference = avg_neighbor_temp - current_temp
    
        #If the neighboring temperature is higher, the pressure decreases.
        #If the neighboring temperature is lower, the pressure increases.
        #based on correlation charts, temp and pressure have a positive correlation.
        pressure_change_from_temp = -0.5 * temp_difference  # Scale factor
    
        # Rule 2: Wind convergence/divergence affects pressure
        #Each cell has wind direction and speed.
        #It calculates the average wind components (east-west and north-south) from all the neighboring cells.
        #cell.wind_direction[0] represents the x-component of wind direction.
        #cell.wind_direction[1] represents the y-component of wind direction.
        #Multiplying wind speed by wind direction gives the magnitude of wind movement in each direction.
        wind_x = sum(cell.weather_data["totalwindspeed"] * (cell.weather_data["windspeedeast"] / cell.weather_data["totalwindspeed"] if cell.weather_data["totalwindspeed"] != 0 else 0) for cell in neighbors) / len(neighbors)
        wind_y = sum(cell.weather_data["totalwindspeed"] * (cell.weather_data["windspeedeast"] / cell.weather_data["totalwindspeed"] if cell.weather_data["totalwindspeed"] != 0 else 0) for cell in neighbors)

        #Wind divergence tells us whether air is spreading out or converging.
        #If wind_x and wind_y both add up to a positive number, it means air is spreading apart (divergence).
        #If wind_x and wind_y add up to a negative number, it means air is coming together (convergence).        
        wind_divergence = wind_x + wind_y  

        #If air is spreading apart (wind_divergence > 0), it causes air to sink → Higher pressure.
        #Air moves away from a region (spreads out).
        #This creates a "gap," and air from above sinks down to fill the space.
        #Sinking air increases surface pressure because air is being pushed down.
        #This process leads to clear skies and calm weather since sinking air prevents cloud formation
        #If air is converging (wind_divergence < 0), it forces air to rise → Lower pressure.
        #Airflows meet in one location (they "converge").
        #The air has nowhere to go but up (rises).
        #Rising air reduces surface pressure because air is being lifted away.
        #This process often leads to cloud formation and storms because rising air cools, causing condensation.
        #The scaling factors (0.3 and 0.5) control the effect of wind divergence/convergence on pressure. 
        #In some weather models, convergence can have a 50% stronger impact than divergence on pressure changes, which matches the 0.5 vs. 0.3 ratio in your code.
        pressure_change_from_wind = 0.3 * wind_divergence if wind_divergence > 0 else 0.5 * wind_divergence 
    
        #Adds both contributions (temperature and wind) to update the cell's pressure.
        new_pressure += pressure_change_from_temp + pressure_change_from_wind

        return new_pressure
    
    def update_precipitation(self):
        neighbors = self.neighbors
        avg_temp = np.nanmean([cell.weather_data["temperature"] for cell in neighbors])
        avg_pressure = np.nanmean([cell.weather_data["pressure"] for cell in neighbors])
        avg_humidity = np.nanmean([cell.weather_data["humidity"] for cell in neighbors])

        new_precipitation = self.weather_data["precipitation"]

        if avg_humidity > 70:
            new_precipitation += 0.1
        if avg_pressure < 1000:
            new_precipitation += 0.1
        if avg_temp > 30:
            new_precipitation += 0.05
        
        return new_precipitation
        
    def update_humidity(self):
        neighbors = self.neighbors
        return self.weather_data["humidity"] + 0.1

    def update_wind_speed(self):
        """
        Updates wind speed based on pressure difference, terrain effect, randomness,
        and neighbor smoothing. Simulates realistic wind dynamics.
        """
        neighbors = self.neighbors

        if not neighbors:
            return  # No neighbors, no update possible

        # Step 1: Get average pressure and wind components from neighbors
        avg_pressure = np.nanmean([cell.weather_data["pressure"] for cell in neighbors if not np.isnan(cell.weather_data["pressure"])])
        avg_north = np.nanmean([cell.weather_data["windspeednorth"] for cell in neighbors if not np.isnan(cell.weather_data["windspeednorth"])])
        avg_east = np.nanmean([cell.weather_data["windspeedeast"] for cell in neighbors if not np.isnan(cell.weather_data["windspeedeast"])])

        new_north_wind_speed = self.weather_data["windspeednorth"]
        new_east_wind_speed = self.weather_data["windspeedeast"]

        # Step 2: Initialize current wind if missing
        if np.isnan(self.weather_data["windspeednorth"]):
            new_north_wind_speed = avg_north if not np.isnan(avg_north) else 0
        if np.isnan(self.weather_data["windspeedeast"]):
            new_east_wind_speed = avg_east if not np.isnan(avg_east) else 0

        # Step 3: Wind moves from high to low pressure
        if not np.isnan(avg_pressure) and not np.isnan(self.weather_data["pressure"]):
            pressure_diff = avg_pressure - self.weather_data["pressure"]  # positive = high → low
            new_north_wind_speed += pressure_diff * random.uniform(0.2, 0.5)
            new_east_wind_speed += pressure_diff * random.uniform(0.2, 0.5)

        # Step 4: Apply terrain factor (ocean retains wind better than land)
        terrain_factor = WindSpeedFactor.OCEAN.value if self.is_ocean else WindSpeedFactor.LAND.value
        new_north_wind_speed *= terrain_factor
        new_east_wind_speed *= terrain_factor

        # Step 5: Add natural random fluctuation
        variation = WindSpeedFactor.BASE_WIND_VARIATION.value
        new_north_wind_speed += random.uniform(-variation, variation)
        new_east_wind_speed += random.uniform(-variation, variation)

        # Step 6: Smooth wind by averaging with neighbors
        if not np.isnan(avg_north):
            new_north_wind_speed = (new_north_wind_speed + avg_north) / 2
        if not np.isnan(avg_east):
            new_east_wind_speed = (new_east_wind_speed + avg_east) / 2

        # Step 7: Clamp wind speed to safe range
        new_north_wind_speed = max(-50, min(50, new_north_wind_speed))
        new_east_wind_speed= max(-50, min(50, new_east_wind_speed))

        # Step 8: Compute final total wind speed (magnitude)
        new_total_wind_speed = np.sqrt(new_north_wind_speed**2 + new_east_wind_speed**2)

        return [new_north_wind_speed, new_east_wind_speed, new_total_wind_speed]
    
    def __repr__(self):
        return (
            f"MapGridCell(id={self.unique_id}, "
            f"pos={self.pos}, "
            f"state={self.state}, "
            f"next_state={self.next_state}, "
            f"latitude={self.latitude}, "
            f"longitude={self.longitude}, "
            f"simulation_parameter={self.simulation_parameter}, "
            f"is_ocean={self.is_ocean}, "
            f"temperature={self.weather_data["temperature"]})"
        )

<h1>Grid Model</h1>

Represents a region (930km by 930km area) which in this case covers the Florida state. 

In [7]:
class MapGridModel(Model):
    """Manages the cellular automaton grid and initialization from data using MESA."""
    def __init__(self, selected_date="2024-10-10", selected_time = "00:00", size=30, seed=42, simulation_prm = WeatherParameter.TEMPERATURE, data=None):
        super().__init__(seed=seed)
        self.size = size
        self.grid = MultiGrid(size, size, torus=False)
        self.simulation_parameter = simulation_prm

        #   ------------------------------------
        #   Weather Data
        #   ------------------------------------
        #   Read the data during initialization to avoid having to provide each input for all
        #   30 x 30 cells
        date_filter = pd.to_datetime(selected_date)
        time_filter = pd.to_datetime(selected_time).time()

        # Filter the DataFrame for the selected date and time
        filtered_df = weather_df[(weather_df["valid_time"].dt.date == date_filter.date()) &
                         (weather_df["valid_time"].dt.time == time_filter)]

        # Store the filtered data for use in the simulation
        self.weather_data = filtered_df.copy()
        
        #       def __init__(self, model, cell, is_ocean, latitude, longitude):
        # Initialize each cell in the grid with a state (e.g., 0)
        if self.weather_data.empty:
            for x in range(self.grid.width):
                for y in range(self.grid.height):
                    cell = MapGridCell(
                        self,
                        is_ocean = False,
                        latitude = np.nan,
                        longitude = np.nan,
                    )
                    self.grid.place_agent(cell, (x, y))  # Place the Cell object in the grid
        else:
            self.initialize_grid()

        self.running = True

    def step(self):
        """Perform the model step in two stages:
        - First, all cells calculate their next state
        - Then, all cells change state to their next state.
        """
        self.agents.do("determine_state")
        self.agents.do("assume_state")

    def initialize_grid(self):
        """Maps lat/lon data to the automaton grid, storing real-world coordinates in cells."""
        df_new = self.weather_data.copy()

        # Define lat/lon bin edges
        lat_min, lat_max = df_new["latitude"].min(), df_new["latitude"].max()
        lon_min, lon_max = df_new["longitude"].min(), df_new["longitude"].max()

        self.lat_bins = np.linspace(lat_min, lat_max, self.size + 1)
        self.lon_bins = np.linspace(lon_min, lon_max, self.size + 1)

        # Compute lat/lon center points for each grid cell
        lat_centers = (self.lat_bins[:-1] + self.lat_bins[1:]) / 2
        lon_centers = (self.lon_bins[:-1] + self.lon_bins[1:]) / 2

        # Initialize grid with empty cells
        for x in range(self.grid.width):
            for y in range(self.grid.height):
                cell = MapGridCell(
                    self,
                    is_ocean = is_ocean_cell(lat_centers[x],lon_centers[y]),
                    latitude = lat_centers[y],
                    longitude = lon_centers[x],
                )
                self.grid.place_agent(cell, (x, y))  # Place the Cell object in the grid

        # Assign data points to grid cells
        df_new["lat_idx"] = np.digitize(df_new["latitude"], self.lat_bins, right=True) - 1
        df_new["lon_idx"] = np.digitize(df_new["longitude"], self.lon_bins, right=True) - 1

        df_new = df_new.groupby(["lat_idx", "lon_idx"], as_index=False).agg({
            "Temperature": "mean",
            "Dewpoint": "mean",
            "Surface Pressure": "mean",
            "Mean Sea Level Pressure": "mean",
            "Total Wind Speed": "mean",
            "Humidity(%)": "mean",
            "North Wind Speed": "mean",
            "East Wind Speed": "mean",
            "Convective Precipitation": "mean",
            "Total Precipitation": "mean",
            "Large Scale Precipitation": "mean"
        })

        # Populate grid with data
        for _, row in df_new.iterrows():
            lat_idx, lon_idx = int(row["lat_idx"]), int(row["lon_idx"])
            if 0 <= lat_idx < self.size and 0 <= lon_idx < self.size:
                pos = (lon_idx, lat_idx)
                agents = self.grid.get_cell_list_contents(pos)
                for agent in agents:
                    if isinstance(agent, MapGridCell):
                        #if lat_idx == 14:
                            #print(f"Temperature at position ({lon_idx},{lat_idx}) is {row["Temperature"]}")
                        agent.weather_data["temperature"] = row["Temperature"]
                        agent.weather_data["dewpoint"] = row["Dewpoint"]

                        #   Pick value from wherever is available for now
                        pressure_value = row["Mean Sea Level Pressure"] if pd.notnull(row["Mean Sea Level Pressure"]) else row["Surface Pressure"]
                        #print(f"Row pressure value {pressure_value}")
                        agent.weather_data["pressure"] = pressure_value

                        agent.weather_data["windspeednorth"] = row["North Wind Speed"]
                        agent.weather_data["windspeedeast"] = row["East Wind Speed"]
                        agent.weather_data["totalwindspeed"] = row["Total Wind Speed"]
                        agent.weather_data["humidity"] = row["Humidity(%)"]

                        precipitation_value = row["Convective Precipitation"] if pd.notnull(row["Convective Precipitation"]) else row["Total Precipitation"]
                        agent.weather_data["precipitation"] = precipitation_value

                        agent.initialize_state()
                        #print(agent)

    def generate_folium_map(self):
        """Generates a Folium map with grid overlay."""
        # Initialize Folium map at the center of Florida
        map_center = [np.mean(self.lat_bins), np.mean(self.lon_bins)]
        florida_map = folium.Map(location=map_center, zoom_start=6)

        # Get temperature range for color mapping
        temp_values = [agent.weather_data["temperature"] 
            for x in range(self.grid.width) 
            for y in range(self.grid.height) 
            for agent in self.grid.get_cell_list_contents((x, y))
        ]
        norm = mcolors.Normalize(vmin=min(temp_values), vmax=max(temp_values))
        colormap = mcolors.LinearSegmentedColormap.from_list("temp_colormap", ["blue", "yellow", "red"])

        for x in range(self.grid.width): 
            for y in range(self.grid.height): 
                for agent in self.grid.get_cell_list_contents((x, y)):
                    color = mcolors.to_hex(colormap(norm(agent.weather_data["temperature"])))
                    folium.Rectangle(
                        bounds=[
                            [self.lat_bins[y], self.lon_bins[x]],  # Bottom-left corner
                            [self.lat_bins[y+1], self.lon_bins[x+1]]  # Top-right corner
                        ],
                        color=color,
                        fill=True,
                        fill_color=color,
                        fill_opacity=0.3,
                        popup=folium.Popup(
                            f"<b>Grid Position:</b> {x},{y}<br>"
                            f"<b>Is Over Ocean?:</b> {agent.is_ocean}<br>"
                            f"<b>East Wind Speed:</b> {agent.weather_data["windspeedeast"]} mph<br>"
                            f"<b>North Wind Speed:</b> {agent.weather_data["windspeednorth"]} mph<br>"
                            f"<b>Total Wind Speed:</b> {agent.weather_data["totalwindspeed"]} mph<br>"
                            f"<b>Pressure:</b> {agent.weather_data["pressure"]} mb<br>"
                            f"<b>Temperature:</b> {agent.weather_data["temperature"]}°C<br>"
                            f"<b>Dewpoint:</b> {agent.weather_data["dewpoint"]}%",
                            f"<b>Humidity:</b> {agent.weather_data["humidity"]}%",
                            max_width=300
                        ),
                    ).add_to(florida_map)

        return florida_map
        

In [8]:
def agent_portrayal(agent):
    if agent.simulation_parameter == WeatherParameter.TEMPERATURE:
        colors = [
            "#0d47a1", "#1976d2", "#42a5f5",  # Shades of Blue (≤ 16)
            "#fdd835", "#ffb300", "#ff9800",  # Yellow to Orange (17 - 30)
            "#e53935", "#b71c1c"              # Shades of Red (> 30)
        ]
        # Define corresponding value boundaries
        boundaries = [0, 8, 16, 22, 26, 30, 40, 50]  # Adjust as needed

        # Create a custom colormap and normalization
        cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=len(colors))
        norm = BoundaryNorm(boundaries, cmap.N)

        # Convert state value to color
        rgba_color = cmap(norm(agent.state))  # Get RGBA color
        color_prm = to_hex(rgba_color)  # Convert to hex

    if agent.simulation_parameter == WeatherParameter.PRESSURE:
        # Define colors for each range (must match len(boundaries) - 1)
        colors = [
            "#b71c1c",  # Red for values < 919
            "#8800cc",  # Dark Fuchsia (lower range 919 - 944)
            "#aa00ff",  # Medium Fuchsia (944 - 955)
            "#d500f9",  # Lighter Fuchsia (955 - 965)
            "#6a00ff",  # Transition between fuchsia and blue (965 - 979)
            "#1976d2"   # Blue for values > 979
        ]
        
        # Define boundaries (each range transition)
        boundaries = [0, 919, 944, 955, 965, 979, 1100]  # 7 boundaries → 6 bins
        
        # Create custom colormap and normalization
        cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=len(colors))
        norm = BoundaryNorm(boundaries, cmap.N)

        # Convert the agent's state to an RGBA color
        rgba_color = cmap(norm(agent.state))  # Get RGBA color
        color_prm = to_hex(rgba_color)  # Convert to hex

    if agent.simulation_parameter == WeatherParameter.HUMIDITY:
        color_prm = "green"

    if agent.simulation_parameter == WeatherParameter.PRECIPITATION:
        # Normalize the agent's state to a value between 0 and 1
        norm = Normalize(vmin=0, vmax=100)  # Adjust vmax based on your state's range
        scalar_map = ScalarMappable(norm=norm, cmap="plasma")  # Use 'RdBu' colormap

        # Convert the agent's state to an RGBA color
        color_prm = scalar_map.to_rgba(agent.state)
        color_prm = to_hex(color_prm)
        
    if agent.simulation_parameter == WeatherParameter.WIND:
        colors = [
            "#0d47a1", "#1976d2", "#42a5f5",  # Shades of Blue (< 74, darker as values increase)
            "#fff176", "#ffee58", "#ffeb3b",  # Shades of Yellow (74-95)
            "#fbc02d", "#f9a825", "#f57f17",  # Darker Yellow (96-110)
            "#ff9800", "#fb8c00", "#ef6c00",  # Shades of Orange (111-129)
            "#e65100", "#d84315", "#bf360c",  # Darker Orange (130-156)
            "#b71c1c", "#c62828", "#d32f2f",  # Intense Red (157+)
        ]

        # Define corresponding value boundaries
        boundaries = [0, 50, 65, 74, 85, 95, 100, 110, 120, 129, 140, 156, 170]  

        # Create custom colormap and normalization
        cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=len(colors))
        norm = BoundaryNorm(boundaries, cmap.N)

        # Convert the agent's state to an RGBA color
        rgba_color = cmap(norm(agent.state))  # Get RGBA color
        color_prm = to_hex(rgba_color)  # Convert to hex
    
    return {
        "color": color_prm,
        "marker": "s",
        "size": 25,
        "filled": True,
        "layer": 0,
        "w": 1,
        "h": 1,
    }

def post_process(ax):
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

model_params_original = {
    "seed": {
        "type": "InputText",
        "value": 42,
        "label": "Random Seed",
    },
    "tmp_land_cooling": {
        "type": "SliderFloat",
        "value": 0.1,
        "label": "Temperature Land Cooling Factor",
        "min": 0.0,
        "max": 1.0,
        "step": 0.01,
    },
    "tmp_ocean_cooling": {
        "type": "SliderFloat",
        "value": 0.1,
        "label": "Temperature Ocean Cooling Factor",
        "min": 0.0,
        "max": 1.0,
        "step": 0.01,
    },
    "simulation_prm": {
        "type": "Select",
        "value": 0,
        "label": "Simulation Parameter",
        "values": [
        (0, "Temperature"),
        (1, "Pressure"),
        (2, "Precipitation"),
        (3, "Wind"),
        (4, "Humidity")
        ]
    }
}

model_params = {
    "seed": {
        "type": "InputText",
        "value": 42,
        "label": "Random Seed",
    },
    "simulation_prm": {
        "type": "Select",
        "value": 0,
        "label": "Simulation Parameter",
        "values": [
        (0, "Temperature"),
        (1, "Pressure"),
        (2, "Precipitation"),
        (3, "Wind"),
        (4, "Humidity")
        ]
    },
    "selected_date": {
        "type": "InputText",
        "value": "2024-10-10",
        "label": "Select Date (YYYY-MM-DD)",
    },
    "selected_time": {
        "type": "InputText",
        "value": "00:00",
        "label": "Select Time (HH:MM)",
    }
}

In [9]:
weather_model = MapGridModel("2024-10-10","00:00",30,42,WeatherParameter.TEMPERATURE)

In [10]:
SpaceGraph = make_space_component(
    agent_portrayal, post_process=post_process, draw_grid=False
)

In [11]:
page = SolaraViz(
    weather_model,
    components=[SpaceGraph],
    model_params=model_params,
    name="Extreme Weather Simulation",
)
page

c:\Users\damia\AppData\Local\Programs\Python\Python312\Lib\site-packages\mesa\visualization\mpl_space_drawing.py:341: UserWarning: the following fields are not used in agent portrayal and thus ignored: filled, layer, w, h.
  arguments = collect_agent_data(space, agent_portrayal, size=s_default)


Cannot show ipywidgets in text

In [12]:
florida_map = weather_model.generate_folium_map()
florida_map.save("florida_temperature_map.html")  # Save to an HTML file

In [13]:
weather_model.weather_data

,valid_time,latitude,longitude,Dewpoint,Temperature,Mean Sea Level Pressure,Sea Surface Temperature,Surface Pressure,East Wind Speed,North Wind Speed,...,High Vegetation Cover,Low Vegetation Cover,Type of High Vegetation,Type of Low Vegetation,Geopotential,Total Precipitation,Convective Precipitation,Large Scale Precipitation,Humidity(%),Total Wind Speed
22968,2024-10-10,32.0,-88.00,15.37637,25.77700,1011.94875,NaN,1004.87125,0.183294,-14.470593,...,0.988942,0.011058,3,1,594.201660,0.000000,0.000000,0.000000,52.677488,14.471754
22969,2024-10-10,32.0,-87.75,15.42910,25.29067,1011.77875,NaN,1003.35125,-2.466521,-13.526884,...,0.992065,0.005601,3,0,708.604000,0.000000,0.000000,0.000000,54.403744,13.749920
22970,2024-10-10,32.0,-87.50,14.98574,25.56410,1011.59875,NaN,1003.93125,-4.511225,-12.425888,...,0.947906,0.048926,3,1,644.979000,0.000000,0.000000,0.000000,52.025335,13.219450
22971,2024-10-10,32.0,-87.25,14.71230,25.67544,1011.51125,NaN,1003.66125,-6.370246,-11.268096,...,0.947906,0.048926,3,1,660.783700,0.000000,0.000000,0.000000,50.781141,12.944111
22972,2024-10-10,32.0,-87.00,14.72012,25.43910,1011.48625,NaN,1002.02125,-7.639449,-10.271958,...,0.981760,0.018240,3,1,796.740700,0.000000,0.000000,0.000000,51.524319,12.801340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23920,2024-10-10,25.0,-81.00,26.10293,28.82583,1003.15875,28.06777,1003.03125,26.666152,28.966274,...,0.000000,0.000000,0,0,10.318848,0.001276,0.000421,0.000862,85.272547,39.371674
23921,2024-10-10,25.0,-80.75,26.21035,28.62075,1003.47625,28.05703,1003.38125,25.198159,29.263367,...,0.000000,0.000000,0,13,8.033691,0.005745,0.002768,0.002965,86.841696,38.617248
23922,2024-10-10,25.0,-80.50,26.27676,28.54458,1003.72125,28.55898,1003.63125,24.350568,30.700779,...,0.000000,0.000000,0,0,7.826660,0.008711,0.004712,0.003980,87.569061,39.185303
23923,2024-10-10,25.0,-80.25,26.27285,28.60708,1004.05875,29.18984,1004.03125,21.235451,32.181878,...,0.000000,0.000000,0,0,2.369629,0.011564,0.007143,0.004430,87.231893,38.556681


In [14]:
df_new = weather_model.weather_data.copy()
the_size = 30

# Define lat/lon bin edges
lat_min, lat_max = df_new["latitude"].min(), df_new["latitude"].max()
lon_min, lon_max = df_new["longitude"].min(), df_new["longitude"].max()
lat_bins = np.linspace(lat_min, lat_max, the_size + 1)
lon_bins = np.linspace(lon_min, lon_max, the_size + 1)
# Assign data points to grid cells
df_new["lat_idx"] = np.digitize(df_new["latitude"], lat_bins, right=True) - 1
df_new["lon_idx"] = np.digitize(df_new["longitude"], lon_bins, right=True) - 1


In [15]:
print(lat_max,lat_min)
print(lon_max,lon_min)

32.0 25.0
-80.0 -88.0


In [16]:
df_new["valid_time"].min()

Timestamp('2024-10-10 00:00:00')

In [17]:
def determine_lat_centers(lat_min, lat_max):
    # Define lat/lon bin edges
    lat_bins = np.linspace(lat_min, lat_max, 31)
    # Compute lat/lon center points for each grid cell
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    return lat_centers

def determine_lon_centers(lon_min, lon_max):
    # Define lat/lon bin edges
    lon_bins = np.linspace(lon_min, lon_max, 31)
    # Compute lat/lon center points for each grid cell
    lon_centers = (lon_bins[:-1] + lon_bins[1:]) / 2
    return lon_centers



In [18]:
lat_centers = determine_lat_centers(25.0,32.0)
lon_centers = determine_lon_centers(-88.0,-80.0)

In [19]:
lat_centers

array([25.11666667, 25.35      , 25.58333333, 25.81666667, 26.05      ,
       26.28333333, 26.51666667, 26.75      , 26.98333333, 27.21666667,
       27.45      , 27.68333333, 27.91666667, 28.15      , 28.38333333,
       28.61666667, 28.85      , 29.08333333, 29.31666667, 29.55      ,
       29.78333333, 30.01666667, 30.25      , 30.48333333, 30.71666667,
       30.95      , 31.18333333, 31.41666667, 31.65      , 31.88333333])

In [20]:
df_new

,valid_time,latitude,longitude,Dewpoint,Temperature,Mean Sea Level Pressure,Sea Surface Temperature,Surface Pressure,East Wind Speed,North Wind Speed,...,Type of High Vegetation,Type of Low Vegetation,Geopotential,Total Precipitation,Convective Precipitation,Large Scale Precipitation,Humidity(%),Total Wind Speed,lat_idx,lon_idx
22968,2024-10-10,32.0,-88.00,15.37637,25.77700,1011.94875,NaN,1004.87125,0.183294,-14.470593,...,3,1,594.201660,0.000000,0.000000,0.000000,52.677488,14.471754,29,-1
22969,2024-10-10,32.0,-87.75,15.42910,25.29067,1011.77875,NaN,1003.35125,-2.466521,-13.526884,...,3,0,708.604000,0.000000,0.000000,0.000000,54.403744,13.749920,29,0
22970,2024-10-10,32.0,-87.50,14.98574,25.56410,1011.59875,NaN,1003.93125,-4.511225,-12.425888,...,3,1,644.979000,0.000000,0.000000,0.000000,52.025335,13.219450,29,1
22971,2024-10-10,32.0,-87.25,14.71230,25.67544,1011.51125,NaN,1003.66125,-6.370246,-11.268096,...,3,1,660.783700,0.000000,0.000000,0.000000,50.781141,12.944111,29,2
22972,2024-10-10,32.0,-87.00,14.72012,25.43910,1011.48625,NaN,1002.02125,-7.639449,-10.271958,...,3,1,796.740700,0.000000,0.000000,0.000000,51.524319,12.801340,29,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23920,2024-10-10,25.0,-81.00,26.10293,28.82583,1003.15875,28.06777,1003.03125,26.666152,28.966274,...,0,0,10.318848,0.001276,0.000421,0.000862,85.272547,39.371674,-1,26
23921,2024-10-10,25.0,-80.75,26.21035,28.62075,1003.47625,28.05703,1003.38125,25.198159,29.263367,...,0,13,8.033691,0.005745,0.002768,0.002965,86.841696,38.617248,-1,27
23922,2024-10-10,25.0,-80.50,26.27676,28.54458,1003.72125,28.55898,1003.63125,24.350568,30.700779,...,0,0,7.826660,0.008711,0.004712,0.003980,87.569061,39.185303,-1,28
23923,2024-10-10,25.0,-80.25,26.27285,28.60708,1004.05875,29.18984,1004.03125,21.235451,32.181878,...,0,0,2.369629,0.011564,0.007143,0.004430,87.231893,38.556681,-1,29
